**MPNST TE Counting Pilot**

This notebook runs a small pilot before processing the full MPNST RNA-seq dataset.



In [ ]:
#Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(stringr)
library(purrr)

project_root <- "/home/kennes38/ResearchProject"
mpnst_root <- file.path(project_root, "25_MPNST_TE_analysis")
dir.create(mpnst_root, showWarnings = FALSE, recursive = TRUE)
setwd(mpnst_root)
manifest_dir <- file.path(mpnst_root, "manifests")
raw_data_dir <- file.path(mpnst_root, "raw_or_external_data")
counts_dir <- file.path(mpnst_root, "te_counts")
pilot_dir <- file.path(counts_dir, "pilot")
dir.create(pilot_dir, showWarnings = FALSE, recursive = TRUE)

sample_manifest_file <- file.path(manifest_dir, "MPNST_RNAseq_sample_manifest.csv")
file.exists(sample_manifest_file)

In [ ]:
# Load manifest and select pilot samples


if (!file.exists(sample_manifest_file)) {
  stop("Sample manifest does not exist yet. Complete Notebook 25.1 first: ", sample_manifest_file)
}

manifest <- read_csv(sample_manifest_file, show_col_types = FALSE) %>%
  mutate(
    PRC2_status = as.character(PRC2_status),
    file_type = as.character(file_type)
  )

cat("Manifest samples:", nrow(manifest), "\n")
manifest %>% count(PRC2_status, file_type)

pilot_manifest <- manifest %>%
  filter(PRC2_status %in% c("PRC2_loss", "PRC2_intact")) %>%
  group_by(PRC2_status) %>%
  slice_head(n = 2) %>%
  ungroup()

write_csv(pilot_manifest, file.path(pilot_dir, "MPNST_TE_count_pilot_manifest.csv"))

pilot_manifest

In [ ]:
# Check tools and write a local SalmonTE install script

check_tool <- function(tool) {
  path <- Sys.which(tool)
  tibble(tool = tool, found = path != "", path = unname(path))
}

tools_to_check <- c(
  "wget",
  "curl",
  "md5sum",
  "pigz",
  "singularity",
  "git",
  "python3",
  "SalmonTE.py",
  "SalmonTE"
)

tool_check <- purrr::map_dfr(tools_to_check, check_tool)
tool_check

if (!all(c("wget", "md5sum", "git", "python3") %in% tool_check$tool[tool_check$found])) {
  stop("The pilot requires wget, md5sum, git and python3.")
}

# Installs SalomTE into a local Python virtual environment
software_dir <- file.path(mpnst_root, "software")
dir.create(software_dir, showWarnings = FALSE, recursive = TRUE)

salmonte_repo <- file.path(software_dir, "SalmonTE")
salmonte_venv <- file.path(software_dir, "salmonte_venv")
salmonte_install_script <- file.path(software_dir, "install_SalmonTE_local.sh")

install_lines <- c(
  "#!/usr/bin/env bash",
  "set -euo pipefail",
  "",
  "# Local SalmonTE install for the MPNST pilot.",
  "# This stays inside 25_MPNST_TE_analysis/software and does not need sudo.",
  "",
  paste0("SOFTWARE_DIR=", shQuote(software_dir)),
  "REPO_DIR=\"${SOFTWARE_DIR}/SalmonTE\"",
  "VENV_DIR=\"${SOFTWARE_DIR}/salmonte_venv\"",
  "mkdir -p \"${SOFTWARE_DIR}\"",
  "cd \"${SOFTWARE_DIR}\"",
  "",
  "if [ ! -d \"${REPO_DIR}/.git\" ]; then",
  "  git clone https://github.com/hyunhwaj/SalmonTE.git \"${REPO_DIR}\"",
  "else",
  "  echo 'SalmonTE repository already exists; leaving it in place.'",
  "fi",
  "",
  "# Recreate/update a small Python environment just for SalmonTE.",
  "python3 -m venv \"${VENV_DIR}\"",
  "source \"${VENV_DIR}/bin/activate\"",
  "python -m pip install --upgrade pip wheel setuptools",
  "# SalmonTE requires snakemake, docopt and pandas according to its README.",
  "python -m pip install snakemake docopt pandas",
  "",
  "# Quick smoke test. This should print the SalmonTE help/version text.",
  "python \"${REPO_DIR}/SalmonTE.py\" --help >/dev/null",
  "echo 'Local SalmonTE install completed successfully.'",
  "echo \"Use: ${VENV_DIR}/bin/python ${REPO_DIR}/SalmonTE.py\""
)

writeLines(install_lines, salmonte_install_script)
Sys.chmod(salmonte_install_script, mode = "0755")

cat("Local SalmonTE install script written to:\n", salmonte_install_script, "\n")
cat("Run it from a Jupyter terminal if SalmonTE is not already configured.\n")

In [ ]:
# Retrieve ENA FASTQ Links and Build the Pilot Download Script

get_ena_fastq_record <- function(run_accession) {
  ena_url <- paste0(
    "https://www.ebi.ac.uk/ena/portal/api/filereport?",
    "accession=", run_accession,
    "&result=read_run",
    "&fields=run_accession,fastq_ftp,fastq_md5,fastq_bytes",
    "&format=tsv"
  )

  tryCatch(
    readr::read_tsv(ena_url, show_col_types = FALSE),
    error = function(e) {
      stop("ENA query failed for ", run_accession, ": ", conditionMessage(e))
    }
  )
}

ena_run_records <- purrr::map_dfr(pilot_manifest$accession, get_ena_fastq_record)

if (nrow(ena_run_records) != nrow(pilot_manifest)) {
  stop("ENA did not return exactly one record for every pilot run.")
}

# ENA separates paired FASTQ files with semicolons in a single table cell
pilot_fastq_files <- purrr::map_dfr(seq_len(nrow(ena_run_records)), function(i) {
  record <- ena_run_records[i, ]
  ftp <- str_split(record$fastq_ftp, ";", simplify = TRUE) %>% as.character()
  md5 <- str_split(record$fastq_md5, ";", simplify = TRUE) %>% as.character()
  bytes <- str_split(record$fastq_bytes, ";", simplify = TRUE) %>% as.character()

  ftp <- ftp[nzchar(ftp)]
  md5 <- md5[nzchar(md5)]
  bytes <- bytes[nzchar(bytes)]

  if (length(ftp) != 2) {
    stop("Expected two paired FASTQ files for ", record$run_accession)
  }

  sample_dir <- file.path(raw_data_dir, record$run_accession)

  tibble(
    sample_id = record$run_accession,
    read_number = seq_along(ftp),
    download_url = paste0("https://", ftp),
    local_path = file.path(sample_dir, basename(ftp)),
    expected_md5 = md5,
    expected_size_gb = as.numeric(bytes) / 1e9
  )
})

write_csv(
  pilot_fastq_files,
  file.path(pilot_dir, "MPNST_pilot_FASTQ_download_manifest.csv")
)

# Replace the provisional FASTQ paths
pilot_fastq_paths <- pilot_fastq_files %>%
  select(sample_id, read_number, local_path) %>%
  tidyr::pivot_wider(
    names_from = read_number,
    values_from = local_path,
    names_prefix = "fastq_"
  )

pilot_manifest <- pilot_manifest %>%
  select(-fastq_1, -fastq_2) %>%
  left_join(pilot_fastq_paths, by = "sample_id")

write_csv(pilot_manifest, file.path(pilot_dir, "MPNST_TE_count_pilot_manifest.csv"))

# Write resumable shell script
download_script <- file.path(pilot_dir, "run_pilot_FASTQ_download.sh")

download_lines <- c(
  "#!/usr/bin/env bash",
  "set -euo pipefail",
  "",
  "# Download four paired-end pilot runs directly from ENA.",
  "# Safe to rerun: wget -c resumes partial downloads.",
  ""
)

for (i in seq_len(nrow(pilot_fastq_files))) {
  f <- pilot_fastq_files[i, ]
  download_lines <- c(
      download_lines,
      "",
      paste0("echo 'Downloading ", basename(f$local_path), "'"),
      paste0("mkdir -p ", shQuote(dirname(f$local_path))),
      paste("wget -c -O", shQuote(f$local_path), shQuote(f$download_url)),
      paste(
        "echo", shQuote(paste(f$expected_md5, f$local_path)),
        "| md5sum -c -"
      )
    )
}

download_lines <- c(download_lines, "", "echo 'Pilot FASTQ downloads completed successfully.'")
writeLines(download_lines, download_script)
Sys.chmod(download_script, mode = "0755")

cat("Pilot runs:", n_distinct(pilot_fastq_files$sample_id), "\n")
cat("FASTQ files:", nrow(pilot_fastq_files), "\n")
cat("Expected compressed size:", round(sum(pilot_fastq_files$expected_size_gb), 2), "GB\n")
download_script
readLines(download_script)

In [ ]:
# Configure SalmonTE and build pilot counting script

software_dir <- file.path(mpnst_root, "software")
dir.create(software_dir, showWarnings = FALSE, recursive = TRUE)

# Salmon install
local_salmonte_repo_script <- file.path(software_dir, "SalmonTE", "SalmonTE.py")
local_salmonte_python <- file.path(software_dir, "salmonte_venv", "bin", "python")


if (file.exists(local_salmonte_repo_script) && file.exists(local_salmonte_python)) {
  salmonte_prefix <- paste(shQuote(local_salmonte_python), shQuote(local_salmonte_repo_script))
  salmonte_mode <- "local virtual environment"
} else if (nzchar(path_salmonte)) {
  salmonte_prefix <- shQuote(path_salmonte)
  salmonte_mode <- "PATH executable"
} else if (file.exists(salmonte_sif) && nzchar(Sys.which("singularity"))) {
  salmonte_prefix <- paste(
    "singularity exec",
    "--bind", shQuote(paste0(mpnst_root, ":", mpnst_root)),
    shQuote(salmonte_sif),
    "SalmonTE.py"
  )
  salmonte_mode <- "Singularity image"
} else {
  salmonte_prefix <- NA_character_
  salmonte_mode <- "not configured"
}

cat("SalmonTE mode:", salmonte_mode, "\n")

# Verify that all eight pilot FASTQ files exist and are non-empty
fastq_check <- pilot_fastq_files %>%
  mutate(
    exists = file.exists(local_path),
    observed_size_gb = ifelse(exists, file.info(local_path)$size / 1e9, NA_real_)
  )

print(fastq_check %>% select(sample_id, read_number, exists, observed_size_gb))

if (!all(fastq_check$exists)) {
  stop("Pilot FASTQs are incomplete. Run the download script before this cell.")
}

if (is.na(salmonte_prefix)) {
  stop(
    "FASTQs are ready, but SalmonTE is not configured. Run: ",
    file.path(software_dir, "install_SalmonTE_local.sh"),
    " from a Jupyter terminal, or place a tested image at: ", salmonte_sif
  )
}

salmonte_script <- file.path(pilot_dir, "run_pilot_SalmonTE.sh")
salmonte_output_dir <- file.path(pilot_dir, "SalmonTE_outputs")

salmonte_lines <- c(
  "#!/usr/bin/env bash",
  "set -euo pipefail",
  "",
  "# SalmonTE pilot for two PRC2-loss and two PRC2-intact MPNST samples.",
  "# Generated by Notebook 25.2 after FASTQ download and SalmonTE setup.",
  "",
  paste0("mkdir -p ", shQuote(salmonte_output_dir))
)

for (i in seq_len(nrow(pilot_manifest))) {
  s <- pilot_manifest[i, ]
  out_i <- file.path(salmonte_output_dir, s$sample_id)
  salmonte_lines <- c(
      salmonte_lines,
      "",
      paste0("echo 'Running SalmonTE for ", s$sample_id, "'"),
      paste0("mkdir -p ", shQuote(out_i)),
      paste(
        salmonte_prefix,
        "quant --reference=hs --outpath", shQuote(out_i),
        shQuote(s$fastq_1),
        shQuote(s$fastq_2)
      )
    )
}

writeLines(salmonte_lines, salmonte_script)
Sys.chmod(salmonte_script, mode = "0755")

salmonte_script
readLines(salmonte_script)

In [ ]:
# Run the download and counting scripts from a terminal

cat("Download script:\n", download_script, "\n")

if (exists("salmonte_script")) {
  cat("SalmonTE script:\n", salmonte_script, "\n")
} else {
  cat("SalmonTE script has not been created yet. Complete Cell 5 after downloading FASTQs.\n")
}


In [ ]:
# Inspect pilot files, outputs and disk usage


list.files(pilot_dir, recursive = TRUE, full.names = TRUE) %>%
  tibble(path = .) %>%
  mutate(size_mb = file.info(path)$size / 1024^2) %>%
  arrange(desc(size_mb)) %>%
  slice_head(n = 30)

cat("Pilot output directory size:\n")
print(system2("du", args = c("-sh", pilot_dir), stdout = TRUE, stderr = TRUE))

cat("Raw pilot FASTQ directory size:\n")
pilot_raw_dirs <- unique(dirname(pilot_manifest$fastq_1))
print(system2("du", args = c("-sh", pilot_raw_dirs), stdout = TRUE, stderr = TRUE))